# AutoShot Phase2: Guarded Resume with Preflight Smoke

Attach DatasetShot, ClipShots, and the previous output dataset at `/kaggle/input/datasets/domanh704/clip-shot-lan-1/autoshotv2`. This notebook patches the cache builder to skip oversized videos, runs a targeted smoke test, then resumes full training.

In [ ]:
from pathlib import Path
import os
import pickle
import shutil
import subprocess
import sys
import warnings

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

# Prefer Kaggle's built-in PyTorch on T4. P100 is sm_60 and may need a cu118 wheel.
gpu_name_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False,
)
gpu_name = gpu_name_result.stdout.strip().split('\n')[0] if gpu_name_result.returncode == 0 else ''
print('Kaggle GPU:', gpu_name or 'not detected')
if 'P100' in gpu_name:
    TORCH_INSTALL = [
        sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir',
        'torch==2.4.1', 'torchvision==0.19.1', 'torchaudio==2.4.1',
        '--index-url', 'https://download.pytorch.org/whl/cu118',
    ]
    print('P100 detected: installing PyTorch 2.4.1 + CUDA 11.8 for sm_60 ...')
    torch_install_result = subprocess.run(TORCH_INSTALL, check=False)
    if torch_install_result.returncode != 0:
        raise RuntimeError('Failed to install PyTorch cu118. Enable Kaggle Internet or switch Accelerator to T4.')
else:
    print('Non-P100 GPU detected: keeping Kaggle built-in PyTorch.')

import torch
print('torch version:', torch.__version__)
print('torch cuda:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
    print('torch arch list:', torch.cuda.get_arch_list())
    if torch.cuda.get_device_capability(0)[0] == 6 and 'sm_60' not in torch.cuda.get_arch_list():
        raise RuntimeError('P100 detected but installed PyTorch still lacks sm_60 support. Switch Accelerator to T4.')

print('Installing ffmpeg-python ...')
pip_result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ffmpeg-python'], check=False)
if pip_result.returncode != 0:
    print('pip install ffmpeg-python failed; notebook will use local fallback shim.')

SHOT_ROOT = Path('/kaggle/input/datasets/domanh704/shotdivide/DatasetShot')
CLIPSHOTS_ROOT = Path('/kaggle/input/datasets/domanh704/dataset-clipshots/ClipShots')
PREFERRED_PREVIOUS_OUTPUT = Path('/kaggle/input/datasets/domanh704/clip-shot-lan-1/autoshotv2')
BAD_TRAIN_KEYS = {
    'clipshots_train:3843913409': 'SIGKILL during decode after shot_train:47465731636',
}
SMOKE_TRAIN_KEYS = [
    'clipshots_train:3843913409',
    'clipshots_train:TheSmashBrothersDocumentarymkv',
]
RUN_SMOKE_TEST = True
RUN_FULL_TRAIN = True
RUN_PREPARE_META = False

def find_dir_named(root, name):
    matches = [p for p in Path(root).rglob(name) if p.is_dir()]
    if not matches:
        raise FileNotFoundError(f'Cannot find directory named {name} under {root}')
    return matches[0]

if not SHOT_ROOT.exists():
    SHOT_ROOT = find_dir_named('/kaggle/input', 'DatasetShot')
if not CLIPSHOTS_ROOT.exists():
    CLIPSHOTS_ROOT = find_dir_named('/kaggle/input', 'ClipShots')

def resume_root_score(root):
    full_cache = root / 'shot_clipshots_phase2_sample_cache.pkl'
    partial_manifest = root / 'shot_clipshots_phase2_sample_cache.pkl.partial.pkl'
    parts_dir = root / 'shot_clipshots_phase2_sample_cache.pkl.parts'
    if not full_cache.exists() and not partial_manifest.exists():
        return None
    part_count = len(list(parts_dir.glob('part_*.pt'))) if parts_dir.exists() else 0
    samples = 0
    if partial_manifest.exists():
        try:
            with partial_manifest.open('rb') as f:
                samples = int(pickle.load(f).get('samples', 0))
        except Exception:
            samples = 0
    return (1 if full_cache.exists() else 0, samples, part_count)

previous_candidates = []
if PREFERRED_PREVIOUS_OUTPUT.exists():
    previous_candidates.append(PREFERRED_PREVIOUS_OUTPUT)
for marker in [
    'shot_clipshots_phase2_sample_cache.pkl',
    'shot_clipshots_phase2_sample_cache.pkl.partial.pkl',
]:
    previous_candidates.extend(p.parent for p in Path('/kaggle/input').rglob(marker))

scored_previous = []
seen_previous = set()
for candidate in previous_candidates:
    if candidate in seen_previous:
        continue
    seen_previous.add(candidate)
    score = resume_root_score(candidate)
    if score is not None:
        scored_previous.append((score, candidate))
if not scored_previous:
    raise FileNotFoundError('No previous output with sample cache was found under /kaggle/input')
PREVIOUS_OUTPUT = max(scored_previous, key=lambda item: item[0])[1]

if (PREVIOUS_OUTPUT / 'pyproject.toml').exists():
    SOURCE_INPUT = PREVIOUS_OUTPUT
else:
    source_candidates = [p.parent for p in Path('/kaggle/input').rglob('pyproject.toml')]
    if not source_candidates:
        raise FileNotFoundError('Attach a source dataset containing pyproject.toml')
    SOURCE_INPUT = sorted(source_candidates, key=lambda p: len(str(p)))[0]

WORK_DIR = Path('/kaggle/working/autoshotv2')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(SOURCE_INPUT, WORK_DIR)
if not (WORK_DIR / 'ckpt_0_200_0.pth').exists():
    ckpt_candidates = list(Path('/kaggle/input').rglob('ckpt_0_200_0.pth'))
    if not ckpt_candidates:
        raise FileNotFoundError('Attach ckpt_0_200_0.pth in source or a checkpoint input dataset')
    shutil.copy2(ckpt_candidates[0], WORK_DIR / 'ckpt_0_200_0.pth')

# Fallback shim only if ffmpeg-python is still unavailable.
try:
    import ffmpeg as _ffmpeg_test
    print('ffmpeg-python import OK')
except Exception:
    print('ffmpeg-python unavailable; writing local fallback ffmpeg.py shim')
    (WORK_DIR / 'ffmpeg.py').write_text(r'''
import subprocess

class _Input:
    def __init__(self, filename):
        self.filename = filename
        self.kwargs = {}

    def output(self, _pipe, **kwargs):
        self.kwargs = kwargs
        return self

    def run(self, capture_stdout=True, capture_stderr=True):
        size = self.kwargs.get('s') or self.kwargs.get('size') or '48x27'
        pix_fmt = self.kwargs.get('pix_fmt', 'rgb24')
        fmt = self.kwargs.get('format', 'rawvideo')
        cmd = ['ffmpeg', '-v', 'error', '-i', self.filename, '-vf', f'scale={size}', '-f', fmt, '-pix_fmt', pix_fmt, 'pipe:1']
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        return proc.stdout, proc.stderr

def input(filename):
    return _Input(filename)
''', encoding='utf-8')

for name in [
    'shot_clipshots_trainval.pickle',
    'shot_clipshots_phase2_sample_cache.pkl',
    'shot_clipshots_phase2_sample_cache.pkl.partial.pkl',
    'shot_clipshots_phase2_sample_cache.pkl.parts',
    'phase2_shot_clipshots_resume.pt',
    'phase2_shot_clipshots_checkpoints',
    'eval_cache_shot_clipshots',
]:
    src = PREVIOUS_OUTPUT / name
    dst = WORK_DIR / name
    if src.exists():
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)

os.chdir(WORK_DIR)
os.environ['PYTHONPATH'] = str(WORK_DIR / 'src')
print('SOURCE_INPUT     =', SOURCE_INPUT)
print('SHOT_ROOT        =', SHOT_ROOT)
print('CLIPSHOTS_ROOT   =', CLIPSHOTS_ROOT)
print('PREVIOUS_OUTPUT  =', PREVIOUS_OUTPUT)
print('PREVIOUS score   =', resume_root_score(PREVIOUS_OUTPUT))
print('WORK_DIR         =', WORK_DIR)

full_cache_path = WORK_DIR / 'shot_clipshots_phase2_sample_cache.pkl'
partial_manifest_path = WORK_DIR / 'shot_clipshots_phase2_sample_cache.pkl.partial.pkl'
parts_dir = WORK_DIR / 'shot_clipshots_phase2_sample_cache.pkl.parts'
parts = sorted(parts_dir.glob('part_*.pt')) if parts_dir.exists() else []
print('RESUME partial   =', partial_manifest_path.exists())
print('RESUME full cache=', full_cache_path.exists())
print('RESUME parts     =', len(parts), parts[-1].name if parts else '')

if not full_cache_path.exists() and partial_manifest_path.exists():
    stale_state = WORK_DIR / 'phase2_shot_clipshots_resume.pt'
    stale_checkpoints = WORK_DIR / 'phase2_shot_clipshots_checkpoints'
    if stale_state.exists():
        try:
            smoke_state = torch.load(stale_state, map_location='cpu', weights_only=False)
            n_samples = smoke_state.get('train_config', {}).get('n_samples', 'unknown')
        except Exception:
            n_samples = 'unknown'
        print('Removing stale train resume before full sample cache is complete:', stale_state, 'n_samples=', n_samples)
        stale_state.unlink()
    if stale_checkpoints.exists():
        print('Removing stale phase2 checkpoints before full sample cache is complete:', stale_checkpoints)
        shutil.rmtree(stale_checkpoints)

if not full_cache_path.exists() and partial_manifest_path.exists() and BAD_TRAIN_KEYS:
    with partial_manifest_path.open('rb') as f:
        manifest = pickle.load(f)
    done_keys = set(manifest.get('done_keys', []))
    skipped = list(manifest.get('skipped', []))
    changed = False
    for bad_key, reason in BAD_TRAIN_KEYS.items():
        if bad_key not in done_keys:
            print('Skipping known bad training video:', bad_key, '-', reason)
            done_keys.add(bad_key)
            skipped.append((bad_key, reason))
            changed = True
    if changed:
        manifest['done_keys'] = sorted(done_keys)
        manifest['skipped'] = skipped
        with partial_manifest_path.open('wb') as f:
            pickle.dump(manifest, f, protocol=pickle.HIGHEST_PROTOCOL)

script_path = WORK_DIR / 'src' / 'autoshotv2' / 'train_phase2.py'
script_text = script_path.read_text(encoding='utf-8')
if '--max-cache-video-frames' not in script_text:
    script_text = script_text.replace('import hashlib\n', 'import hashlib\nimport json\n')
    script_text = script_text.replace('import random\nimport time\n', 'import random\nimport subprocess\nimport time\n')
    guard_code = r'''

def _parse_rate(value: str | None) -> float | None:
    if not value or value in {'0/0', 'N/A'}:
        return None
    if '/' in value:
        num, den = value.split('/', 1)
        den_f = float(den)
        if den_f == 0:
            return None
        return float(num) / den_f
    return float(value)


def probe_video_info(video_path: str) -> dict[str, float | int | None]:
    cmd = [
        'ffprobe', '-v', 'error', '-select_streams', 'v:0',
        '-show_entries', 'stream=nb_frames,avg_frame_rate,r_frame_rate,duration',
        '-show_entries', 'format=duration,size', '-of', 'json', video_path,
    ]
    try:
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
        payload = json.loads(proc.stdout or '{}')
    except Exception as exc:
        return {'probe_error': str(exc), 'duration': None, 'fps': None, 'frames': None, 'size': None}
    stream = (payload.get('streams') or [{}])[0]
    fmt = payload.get('format') or {}
    duration_raw = stream.get('duration') or fmt.get('duration')
    duration = None if duration_raw in {None, 'N/A'} else float(duration_raw)
    fps = _parse_rate(stream.get('avg_frame_rate')) or _parse_rate(stream.get('r_frame_rate'))
    frames_raw = stream.get('nb_frames')
    frames = int(frames_raw) if frames_raw not in {None, 'N/A'} else None
    if frames is None and duration is not None and fps is not None:
        frames = int(duration * fps)
    size_raw = fmt.get('size')
    size = int(size_raw) if size_raw not in {None, 'N/A'} else None
    return {'probe_error': None, 'duration': duration, 'fps': fps, 'frames': frames, 'size': size}


def check_sample_cache_video_budget(video_path: str, args: argparse.Namespace) -> None:
    info = probe_video_info(video_path)
    frames = info.get('frames')
    duration = info.get('duration')
    fps = info.get('fps')
    if frames is not None and args.max_cache_video_frames > 0 and frames > args.max_cache_video_frames:
        raise RuntimeError(
            'video too long for sample cache: '
            f'frames={frames} fps={fps} duration={duration} '
            f'limit_frames={args.max_cache_video_frames} path={video_path}'
        )
    if duration is not None and args.max_cache_video_seconds > 0 and duration > args.max_cache_video_seconds:
        raise RuntimeError(
            'video too long for sample cache: '
            f'duration={duration:.1f}s frames={frames} fps={fps} '
            f'limit_seconds={args.max_cache_video_seconds} path={video_path}'
        )
    if info.get('probe_error'):
        raise RuntimeError(f'ffprobe failed before sample-cache decode: {info["probe_error"]} path={video_path}')
'''
    fn_needle = '\ndef extract_backbone_features(backbone, video_path: str, captured: dict[str, torch.Tensor]) -> torch.Tensor:\n'
    script_text = script_text.replace(fn_needle, guard_code + fn_needle)
    loop_needle = '''            entry = entries[key]
            try:
                feats = extract_backbone_features(backbone, entry["video_path"], captured)
'''
    loop_replacement = '''            entry = entries[key]
            try:
                print(f"  sample cache processing [{i}/{len(work_keys)}] key={key}", flush=True)
                check_sample_cache_video_budget(entry["video_path"], args)
                feats = extract_backbone_features(backbone, entry["video_path"], captured)
'''
    if loop_needle not in script_text:
        raise RuntimeError('Could not patch autoshotv2/train_phase2.py sample-cache loop')
    script_text = script_text.replace(loop_needle, loop_replacement)
    arg_needle = '    parser.add_argument("--max-test-videos", type=int, default=0)\n'
    arg_replacement = arg_needle + '    parser.add_argument("--max-cache-video-frames", type=int, default=180000)\n    parser.add_argument("--max-cache-video-seconds", type=float, default=7200.0)\n'
    script_text = script_text.replace(arg_needle, arg_replacement)
    script_path.write_text(script_text, encoding='utf-8')
    print('Patched sample-cache ffprobe guard into', script_path)

if not full_cache_path.exists() and not (partial_manifest_path.exists() and parts):
    raise RuntimeError('No partial or full sample cache was restored; check the previous output dataset input path.')

In [ ]:
from pathlib import Path
import subprocess

meta_path = Path('/kaggle/working/autoshotv2/shot_clipshots_trainval.pickle')
if meta_path.exists() and not RUN_PREPARE_META:
    print('Using restored metadata:', meta_path)
else:
    subprocess.run([
        'python', '-m', 'autoshotv2.prepare_metadata',
        '--shot-root', str(SHOT_ROOT),
        '--clipshots-root', str(CLIPSHOTS_ROOT),
        '--out', str(meta_path),
        '--val-ratio', '0.10',
        '--max-val-videos', '200',
    ], check=True)

Targeted smoke test. It probes the videos that previously caused `SIGKILL`; if this cell passes, full training starts.

In [ ]:
from pathlib import Path
import pickle
import subprocess

if RUN_SMOKE_TEST:
    with open(meta_path, 'rb') as f:
        smoke_meta = pickle.load(f)

    selected_keys = [key for key in SMOKE_TRAIN_KEYS if key in smoke_meta['entries']]
    for key in smoke_meta['train_keys']:
        if key not in selected_keys:
            selected_keys.append(key)
        if len(selected_keys) >= len(SMOKE_TRAIN_KEYS) + 3:
            break

    smoke_meta = dict(smoke_meta)
    smoke_meta['train_keys'] = selected_keys
    smoke_meta_path = Path('/kaggle/working/autoshotv2/smoke_guard_meta.pickle')
    with smoke_meta_path.open('wb') as f:
        pickle.dump(smoke_meta, f, protocol=pickle.HIGHEST_PROTOCOL)
    print('Smoke train keys:', selected_keys)

    subprocess.run([
        'python', '-m', 'autoshotv2.train_phase2',
        '--meta', str(smoke_meta_path),
        '--base-ckpt', '/kaggle/working/autoshotv2/ckpt_0_200_0.pth',
        '--epochs', '1',
        '--max-total-samples', '0',
        '--max-samples-per-video', '50',
        '--max-val-videos', '1',
        '--max-test-videos', '1',
        '--max-cache-video-frames', '180000',
        '--max-cache-video-seconds', '7200',
        '--sample-cache', '/kaggle/working/autoshotv2/smoke_guard_sample_cache.pkl',
        '--out-ckpt', '/kaggle/working/autoshotv2/smoke_guard_ckpt.pth',
        '--results', '/kaggle/working/autoshotv2/smoke_guard_results.pkl',
        '--eval-cache-dir', '/kaggle/working/autoshotv2/smoke_guard_eval_cache',
        '--rebuild-sample-cache',
        '--no-eval-cache',
        '--no-resume',
        '--skip-test-eval',
    ], check=True)
else:
    print('RUN_SMOKE_TEST=False, skipping smoke test.')

Full training with the same oversized-video guard. `--stop-after-minutes 660` saves progress after about 11 hours, leaving buffer before Kaggle's 12-hour limit.

In [ ]:
if RUN_FULL_TRAIN:
    subprocess.run([
        'python', '-m', 'autoshotv2.train_phase2',
        '--meta', '/kaggle/working/autoshotv2/shot_clipshots_trainval.pickle',
        '--base-ckpt', '/kaggle/working/autoshotv2/ckpt_0_200_0.pth',
        '--sample-cache', '/kaggle/working/autoshotv2/shot_clipshots_phase2_sample_cache.pkl',
        '--resume-state', '/kaggle/working/autoshotv2/phase2_shot_clipshots_resume.pt',
        '--checkpoint-dir', '/kaggle/working/autoshotv2/phase2_shot_clipshots_checkpoints',
        '--out-ckpt', '/kaggle/working/autoshotv2/ckpt_phase2_shot_clipshots_best.pth',
        '--results', '/kaggle/working/autoshotv2/phase2_shot_clipshots_results.pkl',
        '--eval-cache-dir', '/kaggle/working/autoshotv2/eval_cache_shot_clipshots',
        '--epochs', '20',
        '--batch-size', '512',
        '--max-samples-per-video', '160',
        '--max-cache-video-frames', '180000',
        '--max-cache-video-seconds', '7200',
        '--save-every-videos', '25',
        '--save-every-epochs', '1',
        '--log-every-batches', '100',
        '--stop-after-minutes', '660',
    ], check=True)
else:
    print('RUN_FULL_TRAIN=False, skipping full training.')